# FastTranslator: Upgrade Model Pipeline (Drive + HuggingFace)

Notebook ini:
1. Mount Google Drive, ekstrak `fasttranslator-engine.zip` dari `MyDrive/FastTranslator/`
2. Download TinyLlama-1.1B-Chat dari HuggingFace, ekspor ke format `.bin` (llama2.c-compatible)
3. Compile kode C++ dari project yang sudah diekstrak (bukan tulis ulang)
4. Test forward pass fp32 dulu (SEBELUM kuantisasi) -- verifikasi wajib
5. Pack ke `.fllm` INT8 produksi
6. Verifikasi kualitas (perplexity fp32 vs INT8)
7. Simpan `model.fllm` hasil akhir kembali ke Google Drive

**Jalankan cell satu per satu berurutan.** Kalau ada yang error, screenshot cell + errornya,
kirim ke Claude untuk diagnosis.

## 1. Mount Google Drive & Ekstrak Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP_PATH = '/content/drive/MyDrive/FastTranslator/fasttranslator-engine.zip'
assert os.path.exists(ZIP_PATH), f"File tidak ditemukan: {ZIP_PATH} -- cek lagi lokasi upload di Drive Anda"
print("Ditemukan:", ZIP_PATH, "(", os.path.getsize(ZIP_PATH) / 1e6, "MB )")

In [ ]:
!mkdir -p /content/project
!unzip -q -o "{ZIP_PATH}" -d /content/project
!find /content/project -maxdepth 2 -type d

# Auto-detect the actual project root (handles both a zip that contains
# 'fasttranslator-engine/' as a subfolder, or one that extracts flat).
import glob
candidates = glob.glob('/content/project/**/core', recursive=True)
assert candidates, "Tidak ketemu folder 'core/' di dalam zip -- struktur project mungkin beda dari yang diharapkan"
PROJECT_ROOT = os.path.dirname(candidates[0])
print("PROJECT_ROOT =", PROJECT_ROOT)

## 2. Setup Python Dependencies & llama2.c (util publik, bukan kode kita)

In [ ]:
!pip install torch transformers sentencepiece --quiet
!git clone --quiet https://github.com/karpathy/llama2.c.git /content/llama2.c
print("llama2.c siap")

## 3. Download & Ekspor TinyLlama dari HuggingFace

In [ ]:
# Diagnostik: lihat kode permute_reverse asli sebelum di-patch
# (kalau versi export.py berbeda dari yang diasumsikan patch di bawah,
#  screenshot output cell ini dan kirim ke Claude untuk patch presisi)
!sed -n '440,490p' /content/llama2.c/export.py

### Patch: GQA fix untuk `permute_reverse`

`export.py` bawaan llama2.c punya bug untuk model GQA (TinyLlama pakai GQA: `wk`/`wv` cuma
4 head, bukan 32 seperti `wq`). Tanpa patch ini, Cell berikutnya akan gagal dengan error
`RuntimeError: shape '[32, 2, ...]' is invalid for input of size ...`.

Cell di bawah menambal `permute_reverse` supaya sadar `n_kv_heads`. **Aman dijalankan berkali-
kali** -- kalau baris yang dicari tidak ketemu (sudah pernah di-patch, atau versi file beda),
cell akan bilang begitu tanpa merusak apapun.

In [ ]:
path = '/content/llama2.c/export.py'
with open(path) as f:
    src = f.read()

changed = False

# Fix 1: n_kv_heads was hardcoded to num_attention_heads (ignoring real GQA config)
old1 = "    config.n_kv_heads = hf_model.config.num_attention_heads"
new1 = "    config.n_kv_heads = getattr(hf_model.config, 'num_key_value_heads', hf_model.config.num_attention_heads)"
if old1 in src and new1 not in src:
    src = src.replace(old1, new1)
    changed = True
    print('Fix 1 applied: n_kv_heads now reads the real GQA config value')
elif new1 in src:
    print('Fix 1 already applied previously')
else:
    print('Fix 1: exact line not found -- source may differ, check manually')

# Fix 2: wk's permute_reverse call didn't pass GQA-aware n_heads/dim1
old2 = "        layer.attention.wk.weight = nn.Parameter(permute_reverse(hf_dict[f'model.layers.{i}.self_attn.k_proj.weight']))"
new2 = (
    "        kv_dim = config.n_kv_heads * config.dim // config.n_heads\n"
    "        layer.attention.wk.weight = nn.Parameter(permute_reverse(\n"
    "            hf_dict[f'model.layers.{i}.self_attn.k_proj.weight'], config.n_kv_heads, kv_dim, config.dim))"
)
if old2 in src and 'kv_dim = config.n_kv_heads' not in src:
    src = src.replace(old2, new2)
    changed = True
    print('Fix 2 applied: wk permute now uses n_kv_heads-aware shape')
elif 'kv_dim = config.n_kv_heads' in src:
    print('Fix 2 already applied previously')
else:
    print('Fix 2: exact line not found -- source may differ, check manually')

if changed:
    with open(path, 'w') as f:
        f.write(src)
    print()
    print('export.py patched successfully. Re-run the export cell below.')
else:
    print()
    print('No changes written (already patched, or lines not found -- see messages above).')

In [ ]:
%cd /content/llama2.c
!python export.py /content/tinyllama.bin --hf TinyLlama/TinyLlama-1.1B-Chat-v1.0

In [ ]:
# Cari tokenizer.model bawaan checkpoint HF, lalu ekspor ke tokenizer.bin
import glob
tok_candidates = glob.glob('/root/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/**/tokenizer.model', recursive=True)
print("tokenizer.model ditemukan di:", tok_candidates)
if tok_candidates:
    !python tokenizer.py --tokenizer-model="{tok_candidates[0]}"
    !ls -la tokenizer.bin
else:
    print("TIDAK KETEMU tokenizer.model -- jalankan cell berikut untuk cari manual:")
    !find /root/.cache/huggingface -iname '*tokenizer*'

## 4. Verifikasi Config Hasil Ekspor (WAJIB dicek sebelum lanjut)

In [ ]:
import struct, os
size_gb = os.path.getsize("/content/tinyllama.bin") / 1e9
print(f"ukuran file: {size_gb:.2f} GB")
with open("/content/tinyllama.bin", "rb") as f:
    dim, hidden, n_layers, n_heads, n_kv_heads, vocab, seq = struct.unpack("<7i", f.read(28))
    print(f"dim={dim} hidden={hidden} n_layers={n_layers} n_heads={n_heads} n_kv_heads={n_kv_heads} vocab={vocab} seq_len={seq}")
    print()
    print("Ekspektasi TinyLlama-1.1B: dim=2048 n_layers=22 n_heads=32 n_kv_heads=4 vocab=32000")
    if n_kv_heads < n_heads:
        print("OK: n_kv_heads < n_heads terkonfirmasi -- GQA terdeteksi, engine kita sudah siap untuk ini.")
    else:
        print("PERHATIAN: n_kv_heads == n_heads -- bukan GQA, cek lagi apakah model yang ter-export benar.")

**Screenshot output cell di atas sebelum lanjut** kalau ragu -- kirim ke Claude untuk konfirmasi
angka-angkanya sesuai ekspektasi.

## 5. Compile Kode C++ dari Project (dari Drive, bukan tulis ulang)

In [ ]:
TOOLS = os.path.join(PROJECT_ROOT, 'tools')
CORE = os.path.join(PROJECT_ROOT, 'core')
%cd {TOOLS}

!g++ -std=c++17 -O2 -I {CORE}/include \
  {CORE}/src/fllm_parser.cpp {CORE}/src/bitnet_kernel.cpp \
  pack_fllm.cpp -o /content/pack_fllm

!g++ -std=c++17 -O2 -I {CORE}/include \
  {CORE}/src/transformer.cpp {CORE}/src/tokenizer.cpp \
  {CORE}/tests/test_real_model.cpp -o /content/test_real_model

print("Compile selesai.")

## 6. Test Forward Pass FP32 (SEBELUM Kuantisasi) -- Langkah Paling Penting
Kalau output di bawah ini bukan kalimat bahasa Inggris yang koheren, **JANGAN lanjut** ke cell
berikutnya -- screenshot & kirim ke Claude dulu.

In [ ]:
!/content/test_real_model /content/tinyllama.bin /content/llama2.c/tokenizer.bin

## 7. Pack ke `.fllm` Produksi (INT8)

In [ ]:
!/content/pack_fllm /content/tinyllama.bin /content/llama2.c/tokenizer.bin /content/model.fllm
!ls -la /content/model.fllm

## 8. Verifikasi Kualitas Setelah Kuantisasi (fp32 vs INT8)

In [ ]:
%cd {TOOLS}
!g++ -std=c++17 -O2 -I {CORE}/include \
  {CORE}/src/transformer.cpp {CORE}/src/transformer_bitnet.cpp {CORE}/src/transformer_mixed.cpp \
  {CORE}/src/tokenizer.cpp {CORE}/src/fllm_parser.cpp {CORE}/src/kv_cache.cpp \
  eval_perplexity.cpp -o /content/eval_perplexity

print("--- fp32 baseline ---")
!/content/eval_perplexity fp32 /content/tinyllama.bin /content/llama2.c/tokenizer.bin
print("--- packed .fllm (INT8) ---")
!/content/eval_perplexity int8all /content/model.fllm

**Bandingkan kedua angka perplexity di atas.** Kalau selisihnya kecil (seperti stories15M: 27.11
vs 27.01), kuantisasi berhasil nyaris lossless. Kalau selisihnya besar, kirim hasilnya ke Claude
sebelum lanjut ke device -- mungkin perlu penyesuaian skema kuantisasi untuk model sebesar ini.

## 9. Simpan `model.fllm` ke Google Drive

In [ ]:
import shutil
dest = '/content/drive/MyDrive/FastTranslator/model.fllm'
shutil.copy('/content/model.fllm', dest)
print("Tersimpan di:", dest)
print("Ukuran:", os.path.getsize(dest) / 1e6, "MB")
print()
print("Langkah selanjutnya: buka Google Drive di HP, download model.fllm, lalu:")
print("  adb push model.fllm /sdcard/Android/data/com.fasttranslator/files/models/model.fllm")